In [1]:
!pip install -q datasets sentence-transformers scikit-learn

import numpy as np, pandas as pd, torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load 5,000 reviews
ds = load_dataset("Yelp/yelp_review_full")
df = pd.DataFrame(ds["test"].shuffle(seed=42).select(range(5000)))
df["stars"] = df["label"] + 1

# 2. Turn each review into a vector that captures its meaning
device = "cuda" if torch.cuda.is_available() else "cpu"
st = SentenceTransformer("all-MiniLM-L6-v2", device=device)
emb = st.encode(df["text"].tolist(), batch_size=128,
                show_progress_bar=True, normalize_embeddings=True)

# 3. Group similar reviews into 8 topics
K = 8
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(emb)
df["topic"] = km.labels_

# 4. Describe each topic by its most distinctive words
docs = [" ".join(df.loc[df.topic == k, "text"]) for k in range(K)]
vec = TfidfVectorizer(stop_words="english", max_df=0.6, sublinear_tf=True)
M = vec.fit_transform(docs)
terms = np.array(vec.get_feature_names_out())

for k in range(K):
    top = terms[M[k].toarray().ravel().argsort()[::-1][:10]]
    sub = df[df.topic == k]
    print(f"Topic {k} | {len(sub)} reviews | avg stars {sub.stars.mean():.2f} "
          f"| 1-2 star share {(sub.stars <= 2).mean():.2f}")
    print("   ", ", ".join(top))

README.md:   0%|          | 0.00/6.72k [00:00<?, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  299MB            

yelp_review_full/train-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

yelp_review_full/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 23.5MB            

yelp_review_full/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Topic 0 | 921 reviews | avg stars 3.62 | 1-2 star share 0.19
    veneto, vogue, grimaldi, zipp, malabar, ravioli, oysters, scarpetta, biaggis, blackberry
Topic 1 | 380 reviews | avg stars 3.35 | 1-2 star share 0.26
    ramen, teriyaki, bento, lechon, japan, naba, soba, msg, vermicelli, kee
Topic 2 | 644 reviews | avg stars 3.15 | 1-2 star share 0.33
    rooms, theatre, audience, titanic, clubs, properties, monorail, bedroom, casinos, wrist
Topic 3 | 702 reviews | avg stars 2.06 | 1-2 star share 0.70
    destihl, wellington, disappeared, nmdmd, alla, awkwardly, oysters, hostesses, crepe, pepito
Topic 4 | 479 reviews | avg stars 1.78 | 1-2 star share 0.81
    insurance, battery, contacts, cushions, ecofresh, fluid, salon, doctor, keys, reschedule
Topic 5 | 385 reviews | avg stars 3.32 | 1-2 star share 0.30
    boba, latte, cupcake, donuts, crepe, papago, frosting, almond, forest, pinkberry
Topic 6 | 975 reviews | avg stars 2.95 | 1-2 star share 0.41
    guacamole, chipotle, glenn, ntb, t

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words="english", min_df=15, max_df=0.5)
X = cv.fit_transform(df["text"])
terms = np.array(cv.get_feature_names_out())
total = np.asarray(X.sum(0)).ravel()
A = total.sum() / K

for k in range(K):
    idx = np.where(df.topic.values == k)[0]
    cnt = np.asarray(X[idx].sum(0)).ravel()
    score = (cnt / cnt.sum()) * np.log(1 + A / total)
    top = terms[score.argsort()[::-1][:10]]
    sub = df.iloc[idx]
    print(f"\nTopic {k} | {len(idx)} reviews | avg stars {sub.stars.mean():.2f}")
    print("  words:", ", ".join(top))
    near = idx[(emb[idx] @ km.cluster_centers_[k]).argsort()[::-1][:2]]
    for j in near:
        print("  >", df.text.iloc[j][:200].replace("\n", " "))


Topic 0 | 921 reviews | avg stars 3.62
  words: food, good, place, great, service, like, just, time, really, nice
  > I whole-heartadly and HIGHLY recommend this restaurant.....the service was perfect and the food was phenominal.....we ate here for my wife's birthday on August 1st and I can't say enough.  I love the 
  > Food and service were rock-solid the night we were there. Serves a big menu of bistro/brasserie fare. Prices are reasonable for what you get. More importantly, the food is expertly prepared and served

Topic 1 | 380 reviews | avg stars 3.35
  words: sushi, food, place, good, chinese, rice, like, thai, restaurant, roll
  > I heard many good things about this place and so was excited to try it out.\n\nMy friends and I ordered the Thai Beef Salad, Asian Chicken Salad, Monsters Meatball and Spaghetti, and Grilled Salmon. W
  > This place has good food but horrible customer service. They used to be wonderful and it was such a treat when I would drive thirty minutes just to

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

name = "sshleifer/distilbart-cnn-12-6"
tk = AutoTokenizer.from_pretrained(name)
sm = AutoModelForSeq2SeqLM.from_pretrained(name).to(device)

def summarize(text):
    inp = tk(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    out = sm.generate(**inp, max_length=60, min_length=15, num_beams=4)
    return tk.decode(out[0], skip_special_tokens=True)

for k in [3, 4]:
    idx = np.where((df.topic.values == k) & (df.stars.values <= 2))[0]
    near = idx[(emb[idx] @ km.cluster_centers_[k]).argsort()[::-1][:5]]
    print(f"\nTopic {k}: {len(idx)} reviews rated 1-2 stars")
    for j in near:
        print(" -", summarize(df.text.iloc[j]))

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

...

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            


Topic 3: 493 reviews rated 1-2 stars
 -  The food at Cafe Bellagio was "Mediocre at best" The food was mediocre at best . The lack of manners and a smile is just rude .
 -  Of the 15 tables, 2 were occupied and maybe 7 had dirty dishes on them . I was told I'd have to wait or go to the bar . I ordered beef red curry with no basil or mint leaf . 15 min later I get beef red Curry with basil all over the sum
 -  The food was cold, cold, and there was a horrendous selection of lasagna, stuffed pork dish, stuffing, gravy and some sort of pudding item for dessert . The staff was rude and the food was not replenished enough .
 -  The only thing that I really loved was my strawberry basil lemonade which was perfectly fruity and refreshing and made fresh to order . The service was just okay . Without any alcohol, appetizers, or desserts our meal was over 40 dollars for a bland tiny little lunch .
 -  The food is okay, nothing special but not awful . However, the service is probably the worst I